# 第2日 ノートブック03：OpenCV と距離計測

# Day 2 Notebook 03: OpenCV and Distance Measurement

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. OpenCV で画像を読み込み・表示・保存できる | 1. Read, display and save images with OpenCV |
| 2. グレースケール変換と 2 値化の意味を説明できる | 2. Explain greyscale conversion and thresholding |
| 3. 輪郭（contour）を検出して位置を求められる | 3. Find contours and obtain their positions |
| 4. 2 点間の距離をピクセルで計算できる | 4. Compute the distance between two points in pixels |
| 5. ピクセルをミリメートルに変換し、精度の限界を説明できる | 5. Convert pixels to millimetres and explain the accuracy limits |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 60 分 / 60 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第2日ノートブック01・02 を終えていること。 / You have finished Day 2 Notebooks 01 and 02. |
| 必要なライブラリ / Required libraries | opencv-python, numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第2日 / Day 2](../../COURSE_SYLLABUS.md#day2) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| エッジ・輪郭の検出結果 | Edge and contour results |
| ピクセル→mm の換算結果と、測定誤差の考察 | Pixel-to-mm results with a discussion of measurement error |

## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第2日 / Day 2 |
| この日の順番 / Position today | 3 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 60 分 / 60 minutes |
| 前提 / Prerequisites | `Day02_02_Conditions_Loops_Functions.ipynb` |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第2日 / Day 2](../../COURSE_SYLLABUS.md#day2) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


### 予想して、実行して、説明する / Predict, run, inspect, explain

しきい値は画素の明るさを分ける境目です。面積の条件とは別です。 重要なセルの前に、入力・操作・出力を一文ずつ説明してください。
予想を書いて実行し、値または図を観察します。反転しきい値だけを変え、もう一度予想して実行します。
意図的なエラーと未完成の演習は、その指示に従って取り組みます。

A threshold separates pixel intensities; it is not an area cutoff. Before an important cell, describe its input, operation and output in one sentence each.
Predict, run, inspect the values/image and explain why. Change only the inverse threshold, predict and rerun.
Follow the instructions for deliberately erroneous cells and unsolved exercises.


---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
    ("cv2", "opencv-python"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

#### OpenCV の色の順番に注意
OpenCV は色を **BGR**（青・緑・赤）の順で扱います。`cv2.imread()` で読み込んだ配列は BGR です。
Matplotlib は **RGB**（赤・緑・青）の順で表示します。

BGR の配列をそのまま `plt.imshow()` に渡すと、赤と青が入れ替わって見えます。**BGR の配列は、表示する前に変換してください。**

```python
image_bgr = cv2.imread(path)                             # BGR で読み込まれる
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)   # RGB に変換（新しい配列が返る）
```

`cv2.cvtColor()` は **新しい配列を返します**。元の `image_bgr` は書き換わりません。

**いつ変換が必要か。** 変換が要るのは **BGR の配列** だけです。次のものは、もともと RGB なので変換せずに表示します。

- `PIL.Image.open(...)` で開いた画像（実写のカラー参考写真を含む）
- scikit-image などの RGB 配列

逆に、RGB の配列を `COLOR_BGR2RGB` で変換すると色が入れ替わります。**「この配列は BGR か RGB か」を確認してから変換してください。**

研修全体で、**表示は Matplotlib、処理は OpenCV** と決めます。
`cv2.imshow()` は Jupyter で固まることがあるので使いません。

#### Watch out for OpenCV's colour order
OpenCV stores colours as **BGR** (blue, green, red), and `cv2.imread()` returns a BGR array.
Matplotlib displays **RGB**.

Passing a BGR array to `plt.imshow()` swaps red and blue. **Convert a BGR array before display.**

```python
image_bgr = cv2.imread(path)                             # read as BGR
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)   # convert to RGB (a new array is returned)
```

`cv2.cvtColor()` **returns a new array**; it does not modify `image_bgr`.

**When is the conversion needed?** Only for **BGR arrays**. These are already RGB and are displayed
without conversion:

- images opened with `PIL.Image.open(...)`, including the real colour reference photograph
- RGB arrays from libraries such as scikit-image

Converting an RGB array with `COLOR_BGR2RGB` swaps the colours the other way. **Check whether the array
you hold is BGR or RGB** before converting.

Rules for this course: **OpenCV for processing, Matplotlib for display.** Never use `cv2.imshow()`
in a notebook.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

print("OpenCV version:", cv2.__version__)

def show(images, titles=None, figsize=(14, 4), cmap=None):
    """複数の画像を並べて表示する / show several images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        if image.ndim == 3:
            ax.imshow(image)
        else:
            ax.imshow(image, cmap=cmap or "gray", vmin=0, vmax=255)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

#### 予想：赤が青になるのはなぜ？ / Predict: why does red turn blue?

入力は同じ合成BGR帯です。左は誤ってRGBとして表示、右は変換後です。
実行前に左端の色を予想してください。モデル推論でも写真でもありません。

Input is one synthetic BGR strip: left is incorrectly shown as RGB; right is converted.
Predict the leftmost colour before running. This is neither a photograph nor model output.


In [ ]:
# ▸ この値を変更 / Change this value：指定の値だけを変え、結果を予想します。
# ▸ Change this value: edit only the marked value; predict before running.
bgr_strip = np.zeros((30, 90, 3), dtype=np.uint8)
bgr_strip[:, :30] = [0, 0, 255]
bgr_strip[:, 30:60] = [0, 255, 0]
bgr_strip[:, 60:] = [255, 0, 0]
rgb_strip = cv2.cvtColor(bgr_strip, cv2.COLOR_BGR2RGB)
print("left pixel [B,G,R]:", bgr_strip[0, 0].tolist())
print("left pixel [R,G,B]:", rgb_strip[0, 0].tolist())
show([bgr_strip, rgb_strip], ["WRONG: BGR read as RGB", "CORRECT: BGR -> RGB"])


左端の同じ値 `[0,0,255]` はBGRでは赤、RGB表示では青です。
`cvtColor` はチャネル順を変えます。中央の緑が変わらない理由を説明し、左端を `[0,255,255]` に変えて予想します。

The same `[0,0,255]` means red in BGR but blue when displayed as RGB.
`cvtColor` changes channel order. Explain why green stays green, then predict the result of
changing the left band to `[0,255,255]`.


---

## Part 1　画像を読み込む・保存する / Reading and saving images

#### 3 つの基本操作

| 操作 | 関数 | 戻り値 |
|---|---|---|
| 読み込む | `cv2.imread(path)` | NumPy 配列（BGR）。失敗すると `None` |
| 表示する | `plt.imshow(...)` | — |
| 保存する | `cv2.imwrite(path, image)` | 成功で `True`、失敗で `False` |

> `cv2.imread` は、**ファイルが無いとエラーではなく `None` を返します**。
> 必ず `None` かどうかを確認してください。ここは初心者が最も見落とす点です。

`cv2.imwrite` も **例外ではなく `False` を返す** ことがあります。`False` の原因は 1 つではありません。

| `False` の原因 | 確認すること |
|---|---|
| 保存先フォルダが無い | `output_dir.mkdir(parents=True, exist_ok=True)` を先に実行したか |
| 書き込み権限が無い | そのフォルダに書き込めるか（読み取り専用、共有フォルダ、管理者権限） |
| 拡張子と形式が合わない | `.png` / `.jpg` など、拡張子が保存できる形式か（16 bit 画像を `.jpg` で保存するなど） |
| パス名が不正 | 使えない文字、長すぎるパス |

#### The three basic operations
| Operation | Function | Returns |
|---|---|---|
| read | `cv2.imread(path)` | a BGR NumPy array, or `None` on failure |
| display | `plt.imshow(...)` | — |
| save | `cv2.imwrite(path, image)` | `True` on success, `False` on failure |

`cv2.imread` does **not** raise an error for a missing file; it returns `None`. Always check. This is
the most commonly missed detail for beginners.

`cv2.imwrite` can also **return `False` instead of raising**. A missing folder is only one cause.

| Cause of `False` | What to check |
|---|---|
| the target folder does not exist | did you run `output_dir.mkdir(parents=True, exist_ok=True)` first? |
| no write permission | can you write in that folder (read-only, shared folder, administrator rights)? |
| extension and format disagree | does the extension (`.png`, `.jpg`) match what can be stored (a 16-bit image as `.jpg` fails)? |
| invalid path | unsupported characters, an over-long path |


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
image_path = ASSETS / "defect_samples" / "defect_multi.png"

image_bgr = cv2.imread(str(image_path))

if image_bgr is None:
    print("画像が読めませんでした / could not read:", image_path)
    print("assets\\scripts\\generate_training_assets.py を実行してください。")
else:
    print("読み込み成功 / loaded")
    print("shape (height, width, channels):", image_bgr.shape)
    print("dtype:", image_bgr.dtype)
    print("最初の画素の BGR 値 / first pixel BGR:", image_bgr[0, 0])

    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    show([image_bgr, image_rgb],
         ["BGR as-is (wrong colours) / BGR のまま表示", "converted to RGB / RGB に変換"],
         figsize=(11, 4))

#### 保存する
`cv2.imwrite` で保存します。保存先のフォルダは **先に作る** 必要があります。

`cv2.imwrite` は失敗すると `False` を返します（例外ではありません）。**戻り値を必ず確認** し、`False` のときは、フォルダの有無・書き込み権限・拡張子と画像形式の組み合わせを順に確かめてください。

研修では、**元の素材を上書きしない** ように、結果は `submissions` の中に保存します。

#### Saving
`cv2.imwrite` saves an image. The target folder must **already exist**.

`cv2.imwrite` returns `False` on failure instead of raising. **Always check the return value**; when it
is `False`, check the folder, the write permission and the combination of extension and image format,
in that order.

Never overwrite the original assets: this course writes results into `submissions`.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
output_dir = COURSE_ROOT / "submissions" / "day02"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "opencv_rgb_copy.png"

if image_bgr is not None:
    saved = cv2.imwrite(str(output_path), image_bgr)
    print("保存できたか / saved:", saved)
    print("保存先 / path:", output_path)
    print("ファイルサイズ / size:", output_path.stat().st_size, "bytes")

---

## Part 2　グレースケールと 2 値化 / Greyscale and thresholding

### コードの読み方 / Reading the code

![](../../assets/svg/teach_opencv_pixel_pipeline.svg)

図：しきい値と面積の条件は別です。（計算した合成例・模式図。モデルの実測出力ではありません。）

Figure: Intensity threshold and area filter are different. (computed synthetic / schematic, not measured model output.)

**気づいてほしいこと**

しきい値と面積の条件は別です。 入力と変換後の値を指でたどってください。

**What to notice**

Intensity threshold and area filter are different. Trace the input and transformed values.

**考えてみてください / Try to answer**

画素値がしきい値と等しいと、反転マスクでは何になりますか。

What does inverse threshold output when a pixel equals the threshold?


### 元画像 → マスク（前景） → 輪郭 / original → mask (foreground) → contours

![](../../assets/diagrams/teach_opencv_pipeline.png)

図：同じ合成画像を処理した実際の結果（合成画像・実際の計算）。パネル ② は `THRESH_BINARY_INV`（T=150）のマスク **そのもの** で、画素値が 150 以下の画素が白（前景）、150 より大きい画素が黒（背景）です。パネル ③ はそのマスクから求めた輪郭を元画像に重ねた図です。

Figure: the same synthetic image processed step by step (computed from a synthetic image). Panel ② is
the `THRESH_BINARY_INV` mask **itself** (T=150): pixels at or below 150 are white (foreground) and higher
pixels are black (background). Panel ③ overlays the contours found from that mask on the original image.

**気づいてほしいこと**

3 枚は **同じ 1 枚** から作られています。ここで大切なのは **しきい値** と、**`findContours` に渡すマスクの向き（極性）** です。

このノートブックは `cv2.THRESH_BINARY_INV` を使うので、`findContours` に渡すマスクは次のとおりです。

| グレースケール値 | マスクの値 | 見え方 |
|---|---|---|
| しきい値 **以下**（`g <= T`） | 255 | 白（**前景**。`findContours` が探す画素） |
| しきい値 **より大きい**（`g > T`） | 0 | 黒（背景） |

つまり **暗い欠陥ほど白** になります。パネル ② に表示されているマスクが、**そのまま `findContours` に渡す配列** です。白い前景の境界をたどらせるために、この向きが必要です。向きを取り違える（暗い画素を黒のまま渡す）と、背景の外側の枠を輪郭として数えてしまいます。

**What to notice**

All three panels come from **one** source image. What matters here is the **threshold** and the
**polarity of the mask handed to `findContours`**.

This notebook uses `cv2.THRESH_BINARY_INV`, so the mask passed in is:

| Greyscale value | Mask value | Appearance |
|---|---|---|
| **at or below** the threshold (`g <= T`) | 255 | white (**foreground**: the pixels `findContours` searches) |
| **above** the threshold (`g > T`) | 0 | black (background) |

A dark defect therefore becomes white. The mask displayed in panel ② is **exactly the array passed to
`findContours`**. That polarity is what makes the search follow the boundary of the white foreground;
with the wrong polarity (dark pixels left black) it traces the outside border of the image instead.

**考えてみてください / Try to answer**

- しきい値を上げると、輪郭の数は増えますか、減りますか。この合成画像で実際にどうなるか、下の掃引の図で確かめてください。
- If you raise the threshold, do you get more contours or fewer? Check the measured sweep figure below.


#### 同じ小画像：画素 → マスク → 輪郭 / One tiny image: pixels → mask → contour

合成7×8画像の中央3行×4列を暗くします。T=100の反転2値化では等号も白です。
予想：白画素数と、画素中心を結ぶ輪郭の面積は等しいですか。

Darken the middle 3 rows × 4 columns of a synthetic 7×8 image. Inverse threshold T=100
includes equality as white. Predict whether white-pixel count equals the area enclosed by pixel centres.

| g / 画素値 | 40 | 99 | 100 (=T) | 101 |
|---|---|---|---|---|
| 反転マスク / inverse mask | 255 | 255 | 255 | 0 |


In [ ]:
# ▸ この値を変更 / Change this value：指定の値だけを変え、結果を予想します。
# ▸ Change this value: edit only the marked value; predict before running.
tiny_gray = np.full((7, 8), 200, dtype=np.uint8)
tiny_gray[2:5, 2:6] = 40
_, tiny_mask = cv2.threshold(tiny_gray, 100, 255, cv2.THRESH_BINARY_INV)
tiny_contours, _ = cv2.findContours(tiny_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
white_pixels = cv2.countNonZero(tiny_mask)
geometric_area = sum(cv2.contourArea(c) for c in tiny_contours)
print("mask / マスク:\n", tiny_mask)
print("white pixels / 白画素:", white_pixels)
print("contour area / 輪郭面積:", geometric_area, "px2")
truth_pixels = np.array([[40, 99, 100, 101]], dtype=np.uint8)
print("INV truth table:", cv2.threshold(truth_pixels, 100, 255, cv2.THRESH_BINARY_INV)[1])
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, data, title in zip(axes, [tiny_gray, tiny_mask, tiny_mask],
                           ["source / synthetic", "binary inverse mask", "contour on SAME mask"]):
    ax.imshow(data, cmap="gray", vmin=0, vmax=255, origin="upper")
    ax.set_xticks(range(8))
    ax.set_yticks(range(7))
    ax.set_xlabel("x / column (px)")
    ax.set_ylabel("y / row (px)")
    ax.set_title(title)
for contour in tiny_contours:
    points = contour[:, 0, :]
    closed = np.vstack([points, points[0]])
    axes[2].plot(closed[:, 0], closed[:, 1], "o-", color="orange")
plt.tight_layout()
plt.show()


白画素は3×4=12個、輪郭は画素中心間の幅3×高さ2=6 px²です。
輪郭面積を白画素数と呼んではいけません。暗い領域の列終点6を7へ変えると両者は？
後の2→1→18図は、しきい値による2領域の結合と、別のノイズ領域の出現を分けて読みます。

There are 3×4=12 white pixels; the contour spans 3×2 between pixel centres, giving 6 px².
Contour area is not a white-pixel count. Predict both metrics if the dark slice's column stop changes
from 6 to 7. In the later 2→1→18 figure, distinguish merging two regions from separate noise regions appearing.


#### グレースケールにする
色の情報は、**この演習のように「暗い欠陥を明るさで見つける」場面** では使わないことが多いです。
明るさだけにすると、計算が速く、扱いも簡単になります。
ただし、**色そのものが異常**（変色、焼け、異物の色）である場合は、色を残す必要があります。

```python
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
```

`gray` は 2 次元（高さ, 幅）の配列になります。この変換は **3 チャンネル（BGR）の配列** に使います。もともと 1 チャンネルのグレースケール画像（実写の参考写真など）は、そのまま使えて変換は不要です。

#### Converting to greyscale
Colour information is often unnecessary **when, as in this exercise, dark defects are found by
brightness**. Using brightness only is faster and simpler. When **the colour itself is the anomaly**
(discolouration, burn marks, the colour of a foreign object), the colour channels must be kept.

```python
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
```

`gray` is a two-dimensional array (height, width). This conversion applies to a **three-channel BGR
array**; an image that is already single-channel greyscale (the reference greyscale photograph, for example) needs no
conversion.


#### 2 値化（しきい値処理）
2 値化は、**しきい値と比べて各画素を 2 つの値（0 と 255）に置き換える** 処理です。どちらが白になるかは、指定する方法（`THRESH_BINARY` か `THRESH_BINARY_INV` か）で決まります。

```python
_, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
#                        ↑入力  ↑しきい値 T ↑最大値 ↑「反転」2 値化
```

**しきい値ちょうどの画素は「以下」に含まれます。** 次の表のとおりです。

| 元の明るさ | `THRESH_BINARY` | `THRESH_BINARY_INV` |
|---|---|---|
| しきい値より大きい（`g > T`） | 255（白） | 0（黒） |
| しきい値ちょうど（`g == T`） | 0（黒） | 255（白） |
| しきい値より小さい（`g < T`） | 0（黒） | 255（白） |

このノートブックでは、**暗い欠陥を白い前景** にしたいので `THRESH_BINARY_INV` を使います。こうすると、次の「輪郭検出」は「白い部分を探す」という素直な形になります。`findContours` に渡すのは、この白い前景のマスク **そのもの** です（下のコードでも、`binary` をそのまま渡しています）。

`cv2.threshold` は**新しい配列を返す**ので、入力の `gray`（元の画像）は変わりません。しきい値を変えても元の画像は同じで、変わるのはマスクだけです。

しきい値は、ここでは **人間が選んだ固定値（150）** です。OpenCV には `cv2.THRESH_OTSU`（画像から 1 つ自動で決める）や `cv2.adaptiveThreshold`（場所ごとに決める）もありますが、どれも「その結果でよいか」を人間が確認する作業は残ります。

#### Thresholding
Thresholding compares every pixel with a threshold and replaces it with one of two values (0 and 255).
Which side becomes white is decided by the method you choose, `THRESH_BINARY` or `THRESH_BINARY_INV`.

```python
_, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
#                        ^input  ^thresh T ^maxval ^inverted binary
```

**A pixel exactly at the threshold counts as "at or below".**

| Original brightness | `THRESH_BINARY` | `THRESH_BINARY_INV` |
|---|---|---|
| above the threshold (`g > T`) | 255 (white) | 0 (black) |
| exactly the threshold (`g == T`) | 0 (black) | 255 (white) |
| below the threshold (`g < T`) | 0 (black) | 255 (white) |

This notebook uses `THRESH_BINARY_INV` so that **dark defects become white foreground**, which makes
the later contour search simply "find the white regions". The array passed to `findContours` is that
white-foreground mask **itself** (the code below passes `binary` unchanged).

`cv2.threshold` **returns a new array**, so the input `gray` (the original image) is unchanged: changing the threshold leaves the image alone and changes only the mask.

The threshold here is a **fixed value chosen by a person (150)**. OpenCV also offers
`cv2.THRESH_OTSU` (one value chosen automatically from the image) and `cv2.adaptiveThreshold` (a value
per neighbourhood), but a person still has to check whether the result is acceptable.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if image_bgr is not None:
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)

    threshold_value = 150
    _, binary = cv2.threshold(gray, threshold_value, 255, cv2.THRESH_BINARY_INV)

    print("gray shape   :", gray.shape)
    print("gray min/max :", int(gray.min()), "/", int(gray.max()))
    print("white pixels / 白い画素の数:", int((binary == 255).sum()))

    show([gray, binary], ["greyscale / グレースケール", f"binary (thresh={threshold_value}) / 2値化"],
         figsize=(11, 4))

#### しきい値を変えると何が変わるか
しきい値は **人間が決める数値** です。ここが画像処理の難しさです。

この節の実験は、**管理された背景に付いた暗いマーク** を対象に、しきい値を **手で選んで** います。照明や背景が違えば、同じ値が最適とは限りません。

- しきい値を **上げる**（例 200）→ 「暗い」とみなす範囲が広がる → 背景のムラまで前景に入り、**誤検出が増える**
- しきい値を **下げる**（例 60）→ 前景が減る → 薄い欠陥が残らず、**見逃しが増える**

**しきい値 T で変わるもの / 変わらないもの**

| T で変わる / depends on T | T で変わらない / does not depend on T |
|---|---|
| 2 値化マスク（前景の範囲） | 元画像とグレースケール画像 |
| 前景の画素数 | 校正値（px/mm）と撮影条件 |
| 輪郭の数・面積 | 画像の大きさ（画素数の上限） |

次に、いくつかの値で白い前景の画素の数を数えてみましょう。

#### What changes with the threshold
The threshold is a **number chosen by a human**, and that is the hard part.

The experiment in this section uses **thresholds chosen by hand** for dark marks on a controlled
background. A value that works here need not work under different lighting or on a different
background.

- **raising** it (e.g. 200) widens what counts as "dark", so background mottling becomes foreground ->
  more **false alarms**
- **lowering** it (e.g. 60) shrinks the foreground, so faint defects disappear -> more **misses**

**What depends on the threshold T, and what does not**

| depends on T | does not depend on T |
|---|---|
| the binary mask (the extent of the foreground) | the original and greyscale images |
| the number of foreground pixels | the calibration (px/mm) and the imaging setup |
| the contour count and contour areas | the image size (an upper bound on pixel counts) |

Let us count foreground pixels for several thresholds.


### しきい値を掃引して数える / sweeping the threshold and counting

![](../../assets/diagrams/teach_threshold_sweep.png)

図：この合成画像でしきい値を変えたときの、前景の画素数と輪郭数（実測値）。

Figure: foreground pixel count and contour count measured on this synthetic image as the threshold changes.

**気づいてほしいこと**

縦軸は**画素数**で、mm² に換算した面積ではありません。曲線の形は画像と欠陥で変わります。しきい値を変えても輪郭数が変わらない範囲があることも読み取ってください。

**What to notice**

The vertical axis counts **pixels**, not a calibrated area in mm². The shape of the curve depends on the image and the defect; notice also the ranges where the contour count does not change at all.

**考えてみてください / Try to answer**

- しきい値を上げると、前景の画素数は増えますか、減りますか。
- As the threshold rises, does the foreground pixel count grow or shrink?


### 実写のグレースケール画像で試す / try it on a real greyscale photograph

![](../../assets/external_samples/open_reference/camera.png)

写真：実写のグレースケール画像（scikit-image `camera`、Lav Varshney、CC0 1.0）。0〜255 の 1 チャンネルで、この研修の合成画像より複雑です。

Photograph: a real greyscale image (scikit-image `camera`, Lav Varshney, CC0 1.0). One channel, 0–255, and more complex than this course's synthetic images.

**気づいてほしいこと**

照明のムラ、影、背景の模様が、すべて 0〜255 の数になっています。白い前景として残った部分は「欠陥」ではなく、単に『しきい値より暗い画素』です。この画像に製造上の欠陥はありません。

**What to notice**

Uneven lighting, shadows and background texture all become numbers between 0 and 255. White foreground here does not mean "defect"; it means "darker than the threshold". There is no manufacturing defect in this photograph.

**考えてみてください / Try to answer**

- しきい値を 100 にすると、この写真で白く残るのはどんな場所だと思いますか。予想してから実行してください。
- At a threshold of 100, which parts of this photo do you expect to stay white? Predict before you run it.


### 模様のある背景では、数え方が変わる / repeated texture changes what you count

![](../../assets/external_samples/open_reference/brick.png)

写真：レンガ壁の実写（CC0Textures を scikit-image が加工、CC0 1.0）。同じ模様がくり返される背景の例です。正常な模様で、欠陥ではありません。

Photograph: a real brick wall (CC0Textures, processed by scikit-image; CC0 1.0). An example of a background with a repeating pattern. The pattern is normal, not a defect.

**気づいてほしいこと**

同じ模様がくり返される背景では、しきい値を 1 つ決めると**模様の一部がまとめて**前景になります。輪郭の数は『欠陥の数』ではなく『暗い領域の数』です。正常なばらつきを先に知ることが、欠陥検出の出発点になります。

**What to notice**

With a repeating background, one threshold turns **whole parts of the pattern** into foreground. The contour count is then "number of dark regions", not "number of defects". Knowing the normal variation first is the starting point for defect detection.

**考えてみてください / Try to answer**

- この写真のしきい値を上げていくと、白い前景の面積は増えますか、減りますか。
- As you raise the threshold on this photo, does the white foreground grow or shrink?


### 任意の演習：実写の写真でしきい値の限界を見る / optional: the limits of one threshold on real photographs

上の 2 枚は **正常な写真** です。欠陥の写真ではありません。それでも、しきい値を 1 つ決めると「暗い画素」がまとめて前景になります。影・模様・照明のムラが前景に入ることを確かめてください。この演習は **任意** です（完了チェックリストの必須項目ではありません）。

**やってみること（任意 / optional）**

1. 上の**グレースケールの参考写真**を読み、しきい値 100 で `THRESH_BINARY_INV` をかける。
2. 白く残る場所を **予想してから** 実行して確かめる。
3. 今度は**模様のある参考写真**で同じことをし、前景が「欠陥」ではなく「暗い模様」になっていることを確認する。

```python
# ファイル名は、上の図の Markdown 画像リンク（かっこ内のパス）から写してください。
# Copy the file name from the image link in the figure above (the path in parentheses).
reference_name = "<上の図のファイル名 / the file name in the figure above>"
reference_path = ASSETS / "external_samples" / "open_reference" / reference_name
reference_gray = cv2.imread(str(reference_path), cv2.IMREAD_GRAYSCALE)
_, reference_mask = cv2.threshold(reference_gray, 100, 255, cv2.THRESH_BINARY_INV)
reference_contours, _ = cv2.findContours(reference_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print("前景の画素 / foreground pixels:", int((reference_mask == 255).sum()))
print("輪郭の数 / contours:", len(reference_contours))
```

**考えてみてください / Try to answer:** 白い前景は、いつも欠陥ですか。
Does white foreground always mean a defect?

**気づいてほしいこと**

- 前景の画素数は、画像としきい値で決まります。**物理的な面積（mm²）ではありません。**
- 輪郭の数は「暗い領域の数」で、「欠陥の数」ではありません。
- 1 つの固定しきい値で、照明や模様の違う画像を同じように扱うのは無理があります。だから **正常な見え方を先に知る** ことが欠陥検出の出発点になります。

### Optional: the limits of one threshold on real photographs

The two photographs above are **normal images**, not defect images. Even so, one threshold turns "dark
pixels" into foreground: shadows, texture and uneven lighting all join in. This exercise is **optional**
and is not part of the required completion checklist.

**What to do (optional)**

1. Read the **reference greyscale photograph** above and threshold it at 100 with `THRESH_BINARY_INV`.
2. **Predict** which areas stay white, then run it.
3. Repeat with the **textured reference photograph**: the foreground is "dark texture", not a defect.

**What to notice**

- The foreground pixel count depends on the image and on the threshold. It is **not a physical area in
  mm²**.
- The contour count is the number of dark regions, not the number of defects.
- One fixed threshold cannot treat images with different lighting and texture alike, so knowing the
  **normal appearance first** is the starting point for defect detection.

次のセルは、合成画像に戻って、しきい値を 6 通りに変えたときの前景の画素の数を数えます。
The next cell returns to the synthetic image and counts foreground pixels for six thresholds.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if image_bgr is not None:
    print(f"{'しきい値 / threshold':>22}  {'白い画素 / white pixels':>24}")
    print("-" * 50)
    for value in [60, 100, 150, 180, 200, 220]:
        _, candidate = cv2.threshold(gray, value, 255, cv2.THRESH_BINARY_INV)
        print(f"{value:>22}  {int((candidate == 255).sum()):>24}")

---

#### しきい値を動かして、元画像・マスク・輪郭を同時に見る / moving the threshold: original, mask and contours

3 枚は **同じしきい値** から作られます。片方だけが変わることはありません。パネル 2 枚目は、`findContours` に渡すマスク **そのもの**（白＝前景、`THRESH_BINARY_INV`）です。だから「白い面積」と「輪郭の数」は、同じマスクから出た 2 つの数え方になります。

The three panels all come from **one** threshold value; they cannot change independently. The second
panel is the mask passed to `findContours` **itself** (white = foreground, `THRESH_BINARY_INV`), so the
white area and the contour count are two ways of counting the same mask.

| 予想 / Predict | 見るもの / What to look for |
|---|---|
| しきい値を上げる / raise the threshold | 白い面積は増える？ 減る？ / does the white area grow or shrink? |
| 輪郭の数 / number of contours | 増える？ 減る？ / more or fewer? |
| しきい値 0 / threshold 0 | マスクは全部白？ 全部黒？ / all white or all black? |

ウィジェットが無くても、下のパラメータセルで同じ実験ができます。
The parameter cell below runs the same experiment without a widget.


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
import cv2
import matplotlib.pyplot as plt

threshold_to_try = 128      # ← 0〜255 / 0-255

if "image_bgr" in globals() and image_bgr is not None:
    # グレースケールは 1 回だけ計算して使い回す / compute the greyscale once and reuse it
    grey_image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    # 暗い画素を「白い前景」にする（反転 2 値化）/ dark pixels become white foreground
    _, mask = cv2.threshold(grey_image, threshold_to_try, 255, cv2.THRESH_BINARY_INV)
    # この mask が findContours に渡す配列そのものです / this is exactly the array given to findContours
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    drawn = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB).copy()
    cv2.drawContours(drawn, contours, -1, (255, 0, 0), 2)

    print(f"しきい値 / threshold = {threshold_to_try}")
    print(f"前景（しきい値以下）の割合 / foreground fraction = {(mask > 0).mean():.3f}")
    print(f"見つかった輪郭 / contours found = {len(contours)}")

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    axes[0].set_title("元画像 / original", fontsize=9)
    axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
    axes[1].set_title(f"findContours に渡すマスク / mask passed in (T={threshold_to_try})", fontsize=9)
    axes[2].imshow(drawn)
    axes[2].set_title(f"輪郭 / contours: {len(contours)} 個", fontsize=9)
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()
else:
    print("画像が読み込まれていません。Part 1 を先に実行してください。")
    print("The image is not loaded; run Part 1 first.")


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if not ("image_bgr" in globals() and image_bgr is not None):
    print("画像が読み込まれていません。Part 1 を先に実行してください。")
    print("The image is not loaded; run Part 1 first.")
elif not aivision.widgets_available():
    aivision.explain_widget_requirement()
else:
    import cv2
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    from IPython.display import clear_output, display

    grey_image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)   # 1 回だけ / compute once
    original_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    output = widgets.Output()

    def draw(threshold):
        # 表示するマスクと、findContours に渡すマスクは同じ配列 / shown mask = mask passed in
        _, mask = cv2.threshold(grey_image, threshold, 255, cv2.THRESH_BINARY_INV)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        drawn = original_rgb.copy()
        cv2.drawContours(drawn, contours, -1, (255, 0, 0), 2)
        fig, axes = plt.subplots(1, 3, figsize=(13, 4))
        axes[0].imshow(original_rgb)
        axes[0].set_title("元画像 / original", fontsize=9)
        axes[1].imshow(mask, cmap="gray", vmin=0, vmax=255)
        axes[1].set_title(f"findContours に渡すマスク / mask passed in (T={threshold})", fontsize=9)
        axes[2].imshow(drawn)
        axes[2].set_title(f"輪郭 / contours: {len(contours)} 個", fontsize=9)
        for ax in axes:
            ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout()
        with output:
            clear_output(wait=True)
            plt.show()
            plt.close(fig)

    widgets.interact(draw, threshold=widgets.IntSlider(value=128, min=0, max=255, step=1,
                                                       description="threshold"))
    display(output)


## Part 3　輪郭の検出 / Finding contours

#### 輪郭は「白い部分の境界」
`cv2.findContours` は、2 値化画像の **白い部分の境界（輪郭）** を見つけます。返るのは、境界をたどった点の並びです。輪郭は「領域そのもの」ではなく、「前景の画素を数えた記録」でもありません。

```python
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
```

| 引数 | 意味 |
|---|---|
| `cv2.RETR_EXTERNAL` | 一番外側の輪郭だけを返す |
| `cv2.CHAIN_APPROX_SIMPLE` | 輪郭の点を間引いて軽くする |

各輪郭から、次の情報が得られます。

| 関数 | 得られるもの |
|---|---|
| `cv2.contourArea(c)` | 輪郭が囲む図形の面積（画素² 単位の幾何学的な面積） |
| `cv2.boundingRect(c)` | 外接する四角 (x, y, w, h) |
| `cv2.moments(c)` | 重心の計算に使う値 |

`cv2.contourArea` は境界の点から計算した **図形としての面積** です。前景の画素を数えた数とは一致しないことがあります（細い形や境界の扱いで変わります）。またこの値は **画素²** であり、mm² ではありません。mm² にするには校正と、Part 5 で述べる条件が必要です。

#### A contour is the boundary of a white region
`cv2.findContours` finds the **boundaries of the white parts** of a binary image. What comes back is a
sequence of points along each boundary; a contour is neither the region itself nor a record of how many
foreground pixels the region has.

```python
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
```

| Argument | Meaning |
|---|---|
| `cv2.RETR_EXTERNAL` | return only the outermost contours |
| `cv2.CHAIN_APPROX_SIMPLE` | compress the contour points |

From each contour you can obtain:

| Function | Gives |
|---|---|
| `cv2.contourArea(c)` | the geometric area enclosed by the contour, in pixels² |
| `cv2.boundingRect(c)` | the enclosing rectangle (x, y, w, h) |
| `cv2.moments(c)` | values used to compute the centre |

`cv2.contourArea` is a **geometric area** computed from the boundary points. It need not equal a count
of foreground pixels (thin shapes and boundary handling change the result), and it is in **pixels²**,
not mm². Converting to mm² requires calibration and the conditions described in Part 5.


### つながると数が減る / merging reduces the count

![](../../assets/diagrams/teach_contour_merge.png)

図：近い 2 つの欠陥が、しきい値を上げると 2 領域 → 1 領域 → 18 領域と変わる例（合成画像・実際の計算）。

Figure: two nearby defects go from 2 regions to 1 (merged) to 18 (background noise) as the threshold rises (synthetic, computed).

**気づいてほしいこと**

輪郭は領域の**境界**を表す点の列で、前景画素そのものでも画素数でもありません。近い領域がつながると輪郭数は減ります。面積は `contourArea`（輪郭の幾何学的な面積）と、前景画素を数える方法を区別します。

**What to notice**

A contour is the **boundary** of a region — a sequence of points — not the foreground pixels or their count. Nearby regions that merge reduce the contour count. For area, distinguish `contourArea` (geometric area of the contour) from counting foreground pixels.

**考えてみてください / Try to answer**

- 2 つの汚れが 1 本の線でつながると、輪郭数はどうなりますか。
- If two marks are joined by a thin line, what happens to the contour count?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if image_bgr is not None:
    contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    print("見つかった輪郭の数 / contour count:", len(contours))

    min_area = 5      # 小さすぎるノイズを無視する / ignore tiny noise
    kept = [c for c in contours if cv2.contourArea(c) >= min_area]
    print("面積で絞った後 / after the area filter:", len(kept))
    print()

    annotated = image_bgr.copy()
    for index, contour in enumerate(kept, start=1):
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        cv2.drawContours(annotated, [contour], -1, (0, 0, 255), 1)
        cv2.rectangle(annotated, (x, y), (x + w, y + h), (255, 0, 0), 1)
        print(f"輪郭 {index}: 面積 / area = {area:7.1f} px2, bbox = (x={x}, y={y}, w={w}, h={h})")

    show([cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)],
         ["detected contours (red) and boxes (blue) / 検出した輪郭"],
         figsize=(7, 5))

---

## Part 4　2 点間の距離 / Distance between two points

#### まず図で考えましょう
2 点 A と B の距離を求めたいとします。
A から B へまっすぐ測るのは、画像の上では難しいですね。

そこで、**直角三角形** を作ります。

1. 横方向の差：`dx = x2 - x1`
2. 縦方向の差：`dy = y2 - y1`
3. 斜めの長さ（＝距離）は、三平方の定理で求まります

ここで求まる距離の単位は **px（画素）** です。mm に換算するには、Part 5 の校正が必要です。

下の図を見てください。

#### Think with a picture first
Suppose you want the distance between points A and B. Measuring along the diagonal
directly is awkward on a grid.

So build a **right triangle**:

1. horizontal difference: `dx = x2 - x1`
2. vertical difference: `dy = y2 - y1`
3. the diagonal length (the distance) follows from Pythagoras

The result is a distance in **px (pixels)**. Converting it to mm needs the calibration in Part 5.

Look at the figure.


### 2 点の距離は斜辺 / the distance is a hypotenuse

![](../../assets/diagrams/teach_distance_345.png)

図：dx=300, dy=400, 距離=500 px の直角三角形（すべて計算値）。

Figure: a right triangle with dx = 300, dy = 400 and a distance of 500 px (all computed).

**気づいてほしいこと**

距離は「dx と dy をそれぞれ 2 乗して足し、その平方根」で求めます。3-4-5 は、その計算が整数になる例です。px で求めた距離を mm にするには校正（px/mm）が必要です。

**What to notice**

Distance is found by squaring dx and dy, adding them, then taking the square root. 3-4-5 is a case where that calculation gives whole numbers. A distance measured in px needs calibration (px per mm) to become millimetres.

**考えてみてください / Try to answer**

- dx = 6, dy = 8 のとき、距離は何 px ですか。
- With dx = 6 and dy = 8, what is the distance in px?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from PIL import Image

diagram = Image.open(ASSETS / "diagrams" / "diagram_pixel_distance.png")
plt.figure(figsize=(8, 6))
plt.imshow(diagram)
plt.axis("off")
plt.show()

#### 式の意味
式は次の 1 行です。

```text
距離 = √( dx² + dy² )
```

`dx²` は「dx を 2 乗する」、`√` は「平方根（ルート）」です。
**dx と dy をそれぞれ 2 乗して、足して、その平方根を取ります。**
Python では `** 0.5` または `math.sqrt` を使います。

```python
distance = (dx ** 2 + dy ** 2) ** 0.5
```

> 「2 乗して足して、ルートを取る」——これだけです。
> 公式を暗記する必要はありません。**直角三角形の斜辺** を求めているだけです。

求まる距離の単位は **px** です（画素間の距離なので、物理的な長さの単位ではありません）。mm にするには校正で倍率を掛けます。

#### What the formula means
The formula is one line:

```text
distance = sqrt( dx^2 + dy^2 )
```

`dx ** 2` squares `dx`, and `sqrt` takes the square root. **Square dx and dy, add them, then take the
square root.** In Python use `** 0.5` or `math.sqrt`.

```python
distance = (dx ** 2 + dy ** 2) ** 0.5
```

That is only the hypotenuse of a right triangle; there is nothing to memorise.

The result is a distance in **px**: a distance in image space, not a physical length. Multiply by the
calibrated scale to obtain mm.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
def distance_px(point_a, point_b):
    """2 点間の距離をピクセルで返す / distance in pixels between two points.

    Parameters
    ----------
    point_a, point_b : tuple[int, int]
        (x, y) の画素座標 / pixel coordinates (x, y)
    """
    dx = point_b[0] - point_a[0]
    dy = point_b[1] - point_a[1]
    return (dx ** 2 + dy ** 2) ** 0.5


examples = [
    ((0, 0), (3, 4)),
    ((10, 10), (10, 10)),
    ((100, 50), (160, 130)),
    ((0, 0), (0, 25)),
]

for a, b in examples:
    d = distance_px(a, b)
    print(f"A={a}  B={b}  ->  distance = {d:.2f} px")

### 実行結果の例 / Expected output

この例では次の結果になります（同じ入力なら同じ結果です）。

```text
A=(0, 0)  B=(3, 4)  ->  distance = 5.00 px
A=(10, 10)  B=(10, 10)  ->  distance = 0.00 px
A=(100, 50)  B=(160, 130)  ->  distance = 100.00 px
A=(0, 0)  B=(0, 25)  ->  distance = 25.00 px
```

This example produces the following result (the same input gives the same result).


#### 3-4-5 の確認
`(0,0)` と `(3,4)` の距離は **5** です。
`3² + 4² = 9 + 16 = 25`、`√25 = 5`。計算が合っていますね。

#### Verifying 3-4-5
The distance between `(0,0)` and `(3,4)` is **5**, because `3² + 4² = 25` and
`sqrt(25) = 5`.

---

## Part 5　ピクセルをミリメートルに変換する / Converting pixels to millimetres

### 画素をミリメートルにする / turning pixels into millimetres

![](../../assets/diagrams/teach_pixel_to_mm.png)

図：参照長 100 px = 50.0 mm から倍率 2.00 px/mm を決め、200 px の幅を 100.0 mm に換算する例。仮定したエッジ位置の不確かさの換算（±0.5 px ÷ 2.00 px/mm = ±0.25 mm）も同じ図に書かれています。

Figure: a reference of 100 px = 50.0 mm gives the scale 2.00 px/mm, which converts a 200 px width into
100.0 mm. The same figure converts an assumed edge-location uncertainty
(±0.5 px ÷ 2.00 px/mm = ±0.25 mm); it is not guaranteed accuracy.

**気づいてほしいこと**

倍率は「参照長が何 px で何 mm か」から決まります。倍率が決まれば、あとは割り算です。**単位を必ず書いてください。** 図ではエッジ位置の不確かさを ±0.5 px と仮定し、±0.25 mm に換算します。測定精度の保証ではありません。

**この図は 1 つの平面の上での換算の例です。** 参照長と測りたい長さが同じ平面にあり、レンズ歪みが小さく、倍率が視野全体で一定だという前提があります。実際の測定では、**複数の位置と向き** で確かめ、歪みの補正（または領域ごとの校正）も必要です。

**What to notice**

The scale comes from "how many px is how many mm" on the reference; after that it is division.
**Always write the unit.** A pixel error becomes a mm error scaled by that factor (0.5 px is ±0.25 mm
in the figure).

**This figure is a conversion on one plane.** It assumes the reference and the measured length lie in
the same plane, that lens distortion is small and that the scale is constant across the field of view.
Real measurements are checked at **several positions and directions**, with distortion correction (or
calibration per region).

**考えてみてください / Try to answer**

- 同じ実寸 50.0 mm の参照長が、ズームによって 240 px で写るようになりました。倍率は何 px/mm になりますか。そのとき、同じ 1 px の誤差は mm で大きくなりますか、小さくなりますか。
- The same 50.0 mm reference now spans 240 px after zooming in. What is the new scale in px/mm, and does
  the same 1 px error become a larger or a smaller error in mm?


#### なぜ校正（キャリブレーション）が必要か
画像の距離は **ピクセル** で測られます。
しかし現場で知りたいのは **ミリメートル** です。

同じ 100 mm の物体でも、カメラが遠ければ **小さく**、近ければ **大きく** 写ります。
つまり「1 ピクセルが何 mm か」は、**カメラと物体の位置関係で決まる** のです。

そこで、**長さが分かっている物**（基準スケール）を同じ画面に写して、
ピクセルと mm の比率を求めます。これを **校正** と呼びます。

```text
px_per_mm = 基準のピクセル長 ÷ 基準の実寸(mm)
mm_per_px = 基準の実寸(mm) ÷ 基準のピクセル長
```

#### Why calibration is necessary
Distances in an image are measured in **pixels**, but the shop floor needs **millimetres**.

The same 100 mm object appears **smaller** when the camera is far away and **larger** when it is close:
"how many mm is one pixel" depends on the **camera-to-object geometry**.

So you image an object of known size (a reference scale) and derive the ratio. This is
**calibration**.

```text
px_per_mm = reference length in pixels / reference length in mm
mm_per_px = reference length in mm / reference length in pixels
```


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import json

calibration_path = ASSETS / "diagrams" / "calibration_image.png"
calibration_info_path = ASSETS / "diagrams" / "calibration_ground_truth.json"

calibration_image = cv2.imread(str(calibration_path))

if calibration_image is None:
    print("校正画像がありません / calibration image missing:", calibration_path)
    print("assets\\scripts\\generate_training_assets.py を実行してください。")
else:
    info = json.loads(calibration_info_path.read_text(encoding="utf-8"))
    reference_px = info["reference_px"]
    reference_mm = info["reference_mm"]

    px_per_mm = reference_px / reference_mm
    mm_per_px = reference_mm / reference_px

    print("基準の長さ（画素）/ reference pixels :", reference_px)
    print("基準の長さ（実寸）/ reference mm     :", reference_mm)
    print(f"px/mm = {reference_px} / {reference_mm} = {px_per_mm:.3f}")
    print(f"mm/px = {reference_mm} / {reference_px} = {mm_per_px:.5f}")

    show([cv2.cvtColor(calibration_image, cv2.COLOR_BGR2RGB)],
         ["calibration image with a 100 mm reference bar / 校正用画像"], figsize=(7, 5))

#### 校正の考え方の図
下の図は、**同じ 100 mm の基準** が、カメラの距離によって
何ピクセルで写るかを示しています。

- 遠い：100 mm が 120 px → **1.20 px/mm**
- 近い：100 mm が 240 px → **2.40 px/mm**

つまり、校正値は **その撮影条件でしか使えません**。
カメラを動かしたら、**必ず校正し直す** 必要があります。

**2 つの違う操作を混ぜないでください。**

- **基準そのものを 2 倍の長さにする**（100 mm → 200 mm）：実寸が変わるので、倍率（px/mm）は同じままです。
- **ズームして、基準の写る画素数だけを 2 倍にする**（120 px → 240 px）：実寸は 100 mm のままなので、倍率（px/mm）が 2 倍になります。

校正で変わったのは「1 px が何 mm か」です。どちらの操作をしたのかを必ず区別してください。

#### The idea of calibration
The figure shows the **same 100 mm reference** imaged at two camera distances:

- far: 100 mm occupies 120 px -> **1.20 px/mm**
- near: 100 mm occupies 240 px -> **2.40 px/mm**

A calibration value is therefore valid **only for that exact setup**. Move the camera and you must
calibrate again.

**Do not mix up these two different operations.**

- **doubling the real reference length** (100 mm -> 200 mm): the physical length changes, so the scale
  (px/mm) stays the same;
- **doubling the pixel span by zooming in** (120 px -> 240 px): the physical length is still 100 mm, so
  the scale (px/mm) doubles.

What calibration changed is "how many mm one pixel is". Always say which of the two operations you
performed.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
concept = Image.open(ASSETS / "diagrams" / "diagram_calibration_concept.png")
plt.figure(figsize=(10, 4))
plt.imshow(concept)
plt.axis("off")
plt.show()

#### 測定してみる
校正値を使って、2 点間の距離を mm で求めます。

```text
距離(mm) = 距離(px) × mm_per_px
```

下のセルでは、まず校正画像の基準線の両端を測り、**100 mm に戻るか** を確認します。
ただしこれは「倍率を出した線を、同じ倍率で測り直している」だけなので、**計算の確認** にはなっても、測定の正しさの証明にはなりません。

そこで、**校正に使っていない縦の検査線（50 mm）** も測ります。別の位置・別の向きの線が 50 mm に戻れば、**この合成画像の中で換算が一貫していること** の、独立した確認になります。

**この確認で言えること・言えないこと**

- 言えること：この画像の中では、1 つの倍率で別の線も同じように mm に換算できた（**換算の一貫性の確認**）。
- 言えないこと：視野の隅・いろいろな向き・いろいろな距離でも同じ精度で測れること。2 本の線は、どちらも **同じ平面** に、**画像の座標軸に平行** に描かれた理想的な図形です。

実際の測定では、**複数の位置と向き** で確認し、レンズ歪みや平面の傾きも考えます。

#### Measure something
Use the calibration to express a distance in mm:

```text
distance_mm = distance_px * mm_per_px
```

The next cell first measures the reference bar and checks that it comes back as **100 mm**. That only
checks the arithmetic: it measures the same bar the scale came from.

So it then measures a **separate vertical check bar (50 mm)** that was never used for the scale. A line
at another position and another direction coming back as 50 mm is an independent confirmation that
**the conversion is consistent inside this synthetic image**.

**What this check does and does not show**

- it shows: inside this image, one scale factor also converted a different bar to the right length (a
  **consistency check of the conversion**);
- it does not show: the same accuracy at the edges of the field, in other directions or at other
  distances. Both bars are ideal shapes drawn in the **same plane** and **parallel to the image axes**.

Real metrology checks **several positions and directions**, and also considers lens distortion and tilt
of the surface.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
if calibration_image is not None:
    left_point = (info["bar_x1"], info["bar_y"])
    right_point = (info["bar_x2"], info["bar_y"])

    measured_px = distance_px(left_point, right_point)
    measured_mm = measured_px * mm_per_px

    print("測った 2 点 / measured points:", left_point, right_point)
    print(f"距離 / distance : {measured_px:.2f} px")
    print(f"換算 / in mm    : {measured_mm:.3f} mm")
    print(f"真値 / truth    : {reference_mm} mm")
    print(f"誤差 / error    : {abs(measured_mm - reference_mm):.6f} mm")
    print()
    print("※ これは倍率を出した線を測り直しただけです（確認にはなりません）")
    print("   this measures the same bar the scale came from, so it proves nothing")

    # 独立した確認：校正に使っていない縦の検査線を測る
    # An independent check: measure a bar that was never used for the scale.
    if "check_y1" in info:
        top_point = (info["check_x"], info["check_y1"])
        bottom_point = (info["check_x"], info["check_y2"])
        check_px = distance_px(top_point, bottom_point)
        check_mm = check_px * mm_per_px
        print()
        print("独立した確認 / independent check")
        print("測った 2 点 / measured points:", top_point, bottom_point)
        print(f"距離 / distance : {check_px:.2f} px")
        print(f"換算 / in mm    : {check_mm:.3f} mm")
        print(f"真値 / truth    : {info['check_mm']} mm")
        print(f"誤差 / error    : {abs(check_mm - info['check_mm']):.6f} mm")
        print("この線は倍率の計算に使っていません / this bar was not used to compute the scale")
    else:
        print()
        print("検査線がありません。assets\\scripts\\generate_training_assets.py を実行してください。")
        print("no check bar in the ground truth; regenerate the assets.")

### 実行結果の例 / Expected output

この例では次の結果になります（同じ入力なら同じ結果です）。

```text
測った 2 点 / measured points: (100, 150) (220, 150)
距離 / distance : 120.00 px
換算 / in mm    : 100.000 mm
真値 / truth    : 100.0 mm
誤差 / error    : 0.000000 mm

※ これは倍率を出した線を測り直しただけです（確認にはなりません）
   this measures the same bar the scale came from, so it proves nothing

独立した確認 / independent check
測った 2 点 / measured points: (45, 60) (45, 120)
距離 / distance : 60.00 px
換算 / in mm    : 50.000 mm
真値 / truth    : 50.0 mm
誤差 / error    : 0.000000 mm
この線は倍率の計算に使っていません / this bar was not used to compute the scale
```

This example produces the following result (the same input gives the same result).

この 2 本の線はどちらも、合成画像に描いた理想的な図形です。誤差が 0 になるのは、画素座標が整数で与えられ、線が画像の座標軸に平行だからです。実写では、画素のサンプリング、エッジ位置の推定（サブピクセル）、レンズ歪みなどで誤差が出ます。

Both bars are ideal shapes drawn into a synthetic image. The error is exactly zero because the pixel
coordinates are integers and the bars are parallel to the image axes. In a real photograph, pixel
sampling, edge-position estimation (subpixel) and lens distortion all contribute error.


#### 単一のスケール係数が使える条件
「1 つの px/mm で全体を測る」のは、次の条件が満たされるときだけです。

| 条件 | 満たされないとどうなるか |
|---|---|
| カメラがシート面に **正対** している | 斜めだと、場所によって縮尺が変わる |
| レンズ歪みが小さい（または補正済み） | 画像の端ほど誤差が大きい |
| シートが平らで、カメラから **同じ距離** | 浮きや盛り上がりで縮尺が変わる |
| 視野全体で倍率が一定 | 広角レンズは周辺で縮尺が変わる |

**広角レンズでも、校正された測定は可能です。** 歪みが大きいときは、歪み補正（`cv2.calibrateCamera` で係数を求め、`cv2.undistort` で補正する）を使うか、**領域ごとに校正** して局所的な倍率を使います。だめなのは「補正せずに、1 つの倍率を画面全体に当てはめる」ことです。

つまり、**校正していない画像の測定値は、正しいとは言えません**。
「ピクセルで測った」だけでは、mm の保証はありません。

#### When one scale factor is valid
Using a single px/mm for the whole image is valid only when:

| Condition | If it fails |
|---|---|
| the camera faces the sheet **perpendicularly** | oblique views change the scale across the image |
| lens distortion is small or corrected | error grows towards the edges |
| the sheet is flat and at a **constant distance** | wrinkles or bulges change the scale |
| magnification is constant across the field of view | a wide-angle lens changes scale towards the edges |

**A wide-angle lens does not make calibrated measurement impossible.** When distortion is significant,
correct it (`cv2.calibrateCamera` for the coefficients, then `cv2.undistort`) or **calibrate per
region** and use the local scale. What fails is applying one uncorrected scale factor to the whole
frame.

In short, **measurements on an uncalibrated image cannot be trusted**. "I measured it in pixels"
guarantees nothing in millimetres.


---

## 演習 / Exercises

### 同じ平面の基準と対象 / Same-plane reference and targets

長さの換算を確認してから、独立した校正付き欠陥画像を使います。青い基準線は
しきい値80では前景に入りません。倍率1.2 px/mm、独立検査50 mm、輪郭面積475 / 925 px²、
前景画素数1508が、この合成画像のコードから得られます。
演習の式を完成させたときの計算確認値：475 px² → 329.861111 mm², 925 px² → 642.361111 mm²。

After checking length conversion, use a separate same-plane calibrated defect scene.
Blue reference/check bars are excluded by threshold 80. Computed results are 1.2 px/mm,
independent check 50 mm, contour areas 475 / 925 px² and 1508 foreground pixels.
After completing the exercise formula, check the computed areas: 475 px² → 329.861111 mm², 925 px² → 642.361111 mm².


#### まず単位の式 / Start with the unit equation

`px_per_mm = reference_length_px / reference_length_mm`

`measured_mm = measured_px / px_per_mm`

`area_mm2 = area_px2 / (px_per_mm ** 2)`

同一平面の合成例では120 px / 100 mm = 1.2 px/mm、独立の60 pxは50 mmです。
475/925 px²の輪郭は約329.861/642.361 mm²になります。予想：倍率だけ2倍なら換算面積は？
測定精度の認証ではありません。平面・視点・レンズ歪みも確認します。

In the same-plane synthetic example, 120 px / 100 mm = 1.2 px/mm; an independent 60 px gives 50 mm.
Contour areas 475/925 px² become about 329.861/642.361 mm². Predict the converted areas if only the
assumed px/mm doubles. This is not measurement-accuracy certification; check plane, view and distortion too.


### コードの読み方 / Reading the code

![](../../assets/svg/teach_calibration_same_plane.svg)

図：基準線と対象は同じ画像・平面です。（計算した合成例・模式図。モデルの実測出力ではありません。）

Figure: Reference and targets share one image and plane. (computed synthetic / schematic, not measured model output.)

**気づいてほしいこと**

基準線と対象は同じ画像・平面です。 入力と変換後の値を指でたどってください。

**What to notice**

Reference and targets share one image and plane. Trace the input and transformed values.

**考えてみてください / Try to answer**

倍率を2倍に仮定すると、同じ px² の mm² は何倍ですか。

If assumed px/mm doubles, how does the same px² convert to mm²?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
calibrated_path = ASSETS / "diagrams" / "calibrated_defects.png"
calibrated_info_path = ASSETS / "diagrams" / "calibrated_defects_ground_truth.json"
calibrated_image = cv2.imread(str(calibrated_path))
if calibrated_image is None:
    raise FileNotFoundError(f"校正付き欠陥画像がありません / missing: {calibrated_path}")
calibrated_info = json.loads(calibrated_info_path.read_text(encoding="utf-8"))
reference_points = calibrated_info["reference_points"]
calibrated_px_per_mm = distance_px(*reference_points) / calibrated_info["reference_mm"]
check_length_mm = distance_px(*calibrated_info["check_points"]) / calibrated_px_per_mm
print(f"倍率 / scale: {calibrated_px_per_mm:.3f} px/mm")
print(f"独立検査 / independent check: {check_length_mm:.3f} mm")

calibrated_gray = cv2.cvtColor(calibrated_image, cv2.COLOR_BGR2GRAY)
_, calibrated_mask = cv2.threshold(calibrated_gray, 80, 255, cv2.THRESH_BINARY_INV)
calibrated_contours, _ = cv2.findContours(
    calibrated_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
)
calibrated_contours = sorted(calibrated_contours, key=lambda contour: cv2.boundingRect(contour)[0])
for index, contour in enumerate(calibrated_contours, start=1):
    area_px2 = cv2.contourArea(contour)
    print(f"{index}: 輪郭面積 / contour area = {area_px2:.1f} px2")
print("前景画素数 / foreground pixels:", cv2.countNonZero(calibrated_mask))
show([cv2.cvtColor(calibrated_image, cv2.COLOR_BGR2RGB), calibrated_mask],
     ["same-plane calibrated synthetic image", "target mask (threshold 80)"])


#### 演習 1（基本）：同じ画像の輪郭面積を mm² にする

次のセルでは Part 3 の `kept` ではなく、上の校正付き画像の `calibrated_contours` を使います。
Part 3 の `defect_multi.png` に、別画像から得た倍率を適用してはいけません。
ヒント：`area_px2` を `calibrated_px_per_mm ** 2` で割ります。輪郭の幾何学的面積と
前景の画素数は別です。正方画素、同じ対象平面、垂直視点（または補正済みの透視）、
小さいレンズ歪み、基準線の不確かさを確認します。これは理想的な合成画像です。

#### Exercise 1 (basic): convert areas from the same calibrated image

Use `calibrated_contours`, not Part 3's `kept`. Do not apply a different image's scale to
`defect_multi.png`. Hint: divide `area_px2` by `calibrated_px_per_mm ** 2`.
Geometric contour area differs from foreground-pixel count. Check square pixels, the same
object plane, perpendicular view (or corrected perspective), limited distortion and reference
uncertainty. This is an ideal synthetic scene.

**できたら / Done when:** 両輪郭を px² と mm² で表示します / print both areas in px² and mm².


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
if calibrated_image is not None:
    for index, contour in enumerate(calibrated_contours, start=1):
        area_px2 = cv2.contourArea(contour)
        area_mm2 = 0.0     # <- 計算してください / compute
        print(f"{index}: {area_px2:8.1f} px2  ->  {area_mm2:.4f} mm2")

#### 演習 2（中級）：しきい値を変えて輪郭の数を比べる
しきい値を 100 / 150 / 180 / 200 と変えて、
**見つかった輪郭の数** と **最大の輪郭の面積** を表にまとめてください。
どの値が一番「欠陥だけを捉えて」いますか？理由も書いてください。

#### Exercise 2 (intermediate): compare contour counts for several thresholds
For thresholds 100 / 150 / 180 / 200, record the **number of contours** and the
**largest contour area**. Which value captures only the defects? Explain why.
Done when the table has four rows and you justify one chosen threshold.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
if image_bgr is not None:
    for value in [100, 150, 180, 200]:
        _, test_binary = cv2.threshold(gray, value, 255, cv2.THRESH_BINARY_INV)
        test_contours, _ = cv2.findContours(test_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        areas = [cv2.contourArea(c) for c in test_contours if cv2.contourArea(c) >= 5]
        largest = max(areas) if areas else 0.0
        print(f"threshold={value:>3}  contours={len(areas):>3}  largest={largest:8.1f} px2")

#### 演習 3（発展・任意）：測定誤差を考察する / Exercise 3 (optional challenge): discuss measurement error

0.3 mm の欠陥を 1 px/mm で測るとします。

1. 0.3 mm は何ピクセルになりますか。
2. **画素の番号は整数** ですが、輪郭の重心やエッジの位置は **小数（サブピクセル）** で推定できます。この 2 つの違いを説明してください。
3. 0.3 mm を 3 画素で捉えるには、何 px/mm 以上が必要ですか。
4. 誤差を小さくする方法を 3 つ挙げてください（カメラ・照明・校正の観点で）。

**サンプリング・サブピクセル推定・不確かさは別の話です。** 画素の番号が整数でも、位置の推定値を小数で出せば「0.3 px」を連続量として扱えます。ただし、推定値には不確かさ（ばらつき）が付きます。

#### Exercise 3 (optional challenge): discuss measurement error
Suppose you image a 0.3 mm defect at 1 px/mm.

1. How many pixels does 0.3 mm occupy?
2. **Pixel indices are integers**, but a centroid or an edge position can be estimated at
   **subpixel** resolution. Explain the difference between the two statements.
3. How many px/mm are needed for the defect to cover 3 pixels?
4. List three ways to reduce the error (camera, lighting, calibration).

**Sampling, subpixel estimation and uncertainty are separate issues.** Integer pixel indices do not
prevent a subpixel position estimate, but every estimate carries an uncertainty.

**できたら / Done when:** 4 つすべてに答え、3 には数値が書かれている。
All four answers are written and question 3 states a number.


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
defect_mm = 0.3
current_px_per_mm = 1.0

# 1. 0.3 mm は何ピクセルか / how many pixels
pixels_at_current_scale = 0.0        # <- 計算 / compute
# 2. 整数化するとどうなるか / what happens when rounded to whole pixels
# 3. 3 画素で捉えるのに必要な px/mm / px per mm needed for 3 pixels
required_px_per_mm = 0.0             # <- 計算 / compute
# 4. 誤差を小さくする方法 / ways to reduce the error
#    - 
#    - 
#    - 

print("1:", pixels_at_current_scale)
print("2:", "")
print("3:", required_px_per_mm, "px/mm")
print("4:", "")

---

#### 理解度チェック / Understanding questions

1. OpenCV の画像と Matplotlib の画像で、色の順番はどう違いますか。変換を忘れると何が起きますか。
2. `cv2.imread` が存在しないファイルを読んだとき、エラーになりますか。
3. `THRESH_BINARY` と `THRESH_BINARY_INV` の違いは何ですか。なぜ後者を使いましたか。
4. しきい値を上げすぎると、誤検出と見逃しのどちらが増えますか。
5. 「校正していない画像の測定値は信用できない」のはなぜですか。カメラの距離を例に説明してください。

1. How do colour orders differ between OpenCV and Matplotlib, and what happens if you forget to convert?
2. Does `cv2.imread` raise an error for a missing file?
3. What is the difference between `THRESH_BINARY` and `THRESH_BINARY_INV`, and why did we use the latter?
4. If you raise the threshold too much, do false alarms or misses increase?
5. Why can measurements from an uncalibrated image not be trusted? Use camera distance as an example.
Model answers: `docs/instructor/day02_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `image_bgr` が `None` になる | `image_bgr` is `None` | パスが違うか、素材が未生成です。`ASSETS / "defect_samples" / "defect_multi.png"` を確認してください。 | Wrong path, or the assets were never generated. Check the path. |
| `cv2.imwrite` が `False` を返す | `cv2.imwrite` returns `False` | 保存先フォルダが無い、書き込み権限が無い、拡張子と画像形式が合わない、パス名が不正、の順に確認してください。フォルダは `mkdir(parents=True, exist_ok=True)` で先に作ります。 | Check, in order: the folder exists, you have write permission, the extension matches the format, and the path is valid. Create the folder first with `mkdir(parents=True, exist_ok=True)`. |
| 色がおかしい（赤と青が入れ替わる） | Colours look swapped (red/blue) | BGR の配列に `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)` をかけ忘れています。逆に、もともと RGB の画像（PIL で開いた写真など）を変換すると入れ替わります。 | You forgot `cv2.cvtColor(image, cv2.COLOR_BGR2RGB)` on a BGR array. Conversely, converting an already-RGB image (a photo opened with PIL) swaps the colours. |
| `findContours` が 3 つの値を返すと言われる | `findContours` allegedly returns three values | OpenCV 3 系の情報です。OpenCV 4 系では 2 つ（contours, hierarchy）です。 | That is OpenCV 3 documentation. In OpenCV 4 it returns two values. |
| 輪郭が多すぎる／ノイズだらけ | Too many contours / mostly noise | `cv2.contourArea` で小さい輪郭を捨ててください（例：5 px² 未満）。しきい値と極性（`THRESH_BINARY_INV`）も確認してください。 | Filter small contours with `cv2.contourArea` (e.g. drop below 5 px²), and check the threshold and the polarity (`THRESH_BINARY_INV`). |
| 面積の換算が合わない | The area conversion looks wrong | 面積は `px_per_mm ** 2` で割ります（長さは 1 乗、面積は 2 乗）。この式は、画素が正方形で、対象が校正した面と同じ平面にあるときだけ使えます。 | Divide by `px_per_mm ** 2` (length linear, area quadratic). The formula holds only for square pixels on the same plane as the calibration. |
| 画像の端ほど測定値がずれる（広角レンズ） | Measurements drift near the edges (wide-angle lens) | レンズ歪みです。`cv2.calibrateCamera` と `cv2.undistort` で補正するか、領域ごとに校正してください。広角でも、補正すれば校正された測定はできます。 | That is lens distortion. Correct it with `cv2.calibrateCamera` and `cv2.undistort`, or calibrate per region. A wide-angle lens still allows calibrated measurement once corrected. |
| `ModuleNotFoundError: No module named 'cv2'` | `ModuleNotFoundError: No module named 'cv2'` | カーネルが違うか、`opencv-python` が未導入です。`verify_environment.py` で確認してください。 | Wrong kernel, or `opencv-python` is missing. Run `verify_environment.py`. |


## 完了チェックリスト / Completion checklist

**必須** の項目と **任意** の項目を分けて並べています。**演習 3（発展・任意）** は任意で、未実施でも完了です。時間が余ったときだけ取り組んでください。

Required items and optional items are listed separately. **Exercise 3 (optional challenge)** is
optional; leaving it undone still counts as complete. Do it only if you have spare time.

すべての必須項目にチェックが付いたら、このノートブックは完了です。

This notebook is complete when every required box is checked.

**必須 / required**

- [ ] Part 1〜Part 5 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-5 and ran every code cell
- [ ] 画像を読み込み、保存先に保存できた
      - [ ] Read an image and saved a copy
- [ ] グレースケールと 2 値化の結果を目で確認した
      - [ ] Inspected the greyscale and binary results
- [ ] 輪郭を検出し、面積と外接四角を表示した
      - [ ] Detected contours and printed area and box
- [ ] 2 点間の距離を px で計算できた
      - [ ] Computed a distance in pixels
- [ ] px/mm を計算し、mm に換算できた
      - [ ] Computed px/mm and converted to mm
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

**任意 / optional**

- [ ] 実写の参考写真（グレースケールのものと、模様のあるもの）でしきい値の限界を試した
      - [ ] Tried the threshold-limitation exercise on the photographs
- [ ] 演習 3（発展・任意）の 4 つの問いに答えた
      - [ ] Answered the four questions of Exercise 3 (optional)


## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [OpenCV 公式チュートリアル](https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html)
- [cv2.threshold](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)
- [cv2.findContours](https://docs.opencv.org/4.x/d3/d05/tutorial_py_table_of_contents_contours.html)
- [カメラ校正の考え方](https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

`cv2.adaptiveThreshold` を使ってみてください。照明のムラがある画像で、固定しきい値より良い結果になる理由を考えてください。

Try `cv2.adaptiveThreshold` and explain why it can beat a fixed threshold when the lighting is uneven.